# [LOG04 미션 01] 지속 사용 환경에서의 핵심 관리 물자(주요 소모품, 드론 부품, 보호 자재) 소모율 분석 — 교수자용 해설판 (Workbook)

- **산출물 경로**: `docs/log04_doc_framework.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·물자 제원·재고·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 1. 지속 사용 환경에서의 핵심 관리 물자(주요 소모품, 드론 부품, 보호 자재) 소모율 분석

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「물자 현황 분석표와 기준서 목차」 초안을 만듭니다.

거점마다 물자가 얼마나 남았는지 같은 방식으로 적어야 필요한 곳을 비교할 수 있습니다.

보고의 숫자와 추측을 구분하면 그럴듯하지만 근거 없는 문장을 줄일 수 있습니다.

준비할 자료는 다음과 같습니다: 거점 현황 9행, C1~C5, 보유·소모 대장의 관련 행. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

수량은 단위와 기간을 함께 적고, 계산 결과는 계산기로 다시 확인합니다.

자료에 없는 값은 추측하지 않고 미확인으로 남깁니다.

다른 담당자가 문장과 표를 읽고 같은 자료를 찾을 수 있게 근거를 붙입니다.

### 상황과 자료

가상 조직 A팀과 B팀은 세 거점의 물자를 함께 관리합니다.

현황표 9행과 사례 C1~C5를 받아 물자 사용 속도와 다음 확인 항목을 정리하려 합니다.

**준비할 자료**: 거점 현황 9행, C1~C5, 보유·소모 대장의 관련 행

**뜻 알아두기**: 소모율은 일정 기간에 사용한 양을 기간으로 나눈 값입니다.

보유일수는 현재 양을 하루 사용량으로 나눈 값입니다.

리드타임은 요청 뒤 도착까지 걸리는 시간이며, 여유는 보유일수에서 리드타임을 뺀 값입니다.


In [ ]:
# [환경 준비] 미션 1 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/거점_보유소모_대장.csv': "[교육용가상자료] 거점 보유·소모 대장 — 가상 조직 A팀 3개 거점의 D-3일~D일 기록. 수량 단위는 품목별(발·세트·묶음). 실제 재고·제원과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n거점,품목,단위,D-3일보유량,D-2일소모,D-1일소모,D일소모,입고량_D-2~D,D일보유량_보고,일평균소모량_보고,리드타임_일,보고출처,보고시각,비고\n북곡,AM-1,발,42,6,6,6,0,24,6,3,창고 실사+서명,D일 17:00,D-3일 소모는 3발(이전 수준)\n북곡,DP-2,세트,60,5,5,5,0,40,5,2,불출 대장 집계,D일 17:00,파손 손실 기록란 비어 있음\n북곡,PG-3,묶음,150,10,10,10,0,120,10,1,창고 실사,D일 17:00,\n서림,AM-1,발,27,3,3,3,0,18,3,3,담당자 A 대장 집계,D일 18:00,담당자 B는 소모 5/일로 보고(K4)\n서림,DP-2,세트,24,6,6,0,0,12,4,2,불출 대장 집계,D일 18:00,D일 소모는 18시 집계 전(0으로 기재). 파손 손실과 정상 소모가 한 칸에 합산\n서림,PG-3,묶음,75,15,15,15,0,30,15,1,담당자 구두,D일 18:00,수량 '30 정도'의 어림값\n해솔,AM-1,발,20,4,4,4,0,8,4,3,전일 보고서 인용,D-1일 18:00,전일 인용은 D-1일 18시 출처 시각이다. D일 19시 대장 담당자가 D일 소모 4를 계산용 추정 기입하고 보유량 8을 산출했다. D일 실사값 아님. 현재량 확인 후 사용\n해솔,DP-2,세트,26,2,2,2,0,20,2,2,창고 실사,D일 16:00,\n해솔,PG-3,묶음,78,6,6,6,0,60,6,1,창고 실사,D일 16:00,야간 소모량 미반영\n", '02_데이터셋/재보급_요청대장.csv': "[교육용가상자료] 재보급 요청대장 — D일 가상 조직 A팀이 지원팀에 올린 요청. '요청서기재_보유일수'는 요청 거점이 스스로 적은 값이며 대장 재계산과 다를 수 있다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n요청ID,거점,품목,요청수량,요청시각,요청사유,요청서기재_보유일수,근거유형,수송로상태,도착예정,비고\nQ4,해솔,AM-1,10,D일 07:30,보유일수 2일로 리드타임 3일보다 짧음,2.0,전일 보고서 인용,양호,D+3일,R3와 같은 자료\nQ1,북곡,AM-1,12,D일 09:00,현장 대응 빈도 증가로 소모 6발/일,4.0,창고 실사+서명,양호,D+3일,R1과 같은 자료\nQ3,해솔,PG-3,60,D일 11:00,예방,4.0,요청서 1장(실사·대장 없음),양호,D+1일,C3와 같은 요청. 사유가 '예방'뿐\nQ5,서림,PG-3,30,D일 14:00,일 소모 15묶음,2.0,담당자 구두,양호,D+1일,R4와 같은 자료\nQ6,북곡,DP-2,20,D일 16:00,예방,8.0,근거 없음,양호,D+2일,B4와 같은 요청\nQ2,서림,DP-2,16,D일 18:30,이틀 만에 절반 소진(파손 포함 여부 불명),3.0,불출 대장 집계,지연 1일,D+3일,R2·C2와 같은 자료\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답도, 학습자 수행물도 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n제공 근거 가중치: 보유일수<리드타임 +3, 직전 2일보다 당일 소모 증가 +2, G4·G5·G6 -3, 근거 없는 예방 요청 -2. 모르는 값은?로 별도 기록한다. 종합 점수 = 상향 근거 가중치의 합 − 하향 근거 가중치 절댓값의 합. 규칙1: 점수3 이상은 긴급 승인 검토 후보. 규칙2: G4 이상 또는 독립 교차확인 불일치·미확인이면 보류. 규칙3: 확인 불가 1개 이상이면 추가 확인 요청. 규칙4: 사람의 최종 결정·운영 규칙 우선. 규칙2·3 보류 조건은 점수보다 우선한다. 미션 4의 12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다.\n\n## 미션 6 대체 빈 서식\n\n| 요청ID | 거점 | 품목 | 보유일수 | 여유 | 위험 등급 | 요청수량 | 판단·근거 |\n|---|---|---|---|---|---|---|---|\n| Q1 | | | | | | | |\n| Q2 | | | | | | | |\n| Q3 | | | | | | | |\n| Q4 | | | | | | | |\n| Q5 | | | | | | | |\n| Q6 | | | | | | | |\n\n미션 9에서 필요한 현황은 미션 1·미션 3 결과로 이 빈 서식을 채운다. 미션 6의 메모3종 변환·금지표현6개는 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 3·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 미션 9의 V1~V7에 각각 재적용하고 동일 결과는 근거·유지 이유로 기록한다. 제공된 과거 판단과 내 이전 기준은 분리한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n\n## 사실·판단·조치·미확인 분류\n\n사실은 제공 자료에서 확인한 내용, 판단은 사실을 기준에 적용한 해석, 조치는 다음에 할 행동, 미확인은 근거가 부족한 항목입니다.\n\n| 항목 | 사실과 자료 ID | 판단과 적용 기준 | 조치 | 미확인·재확인 대상 |\n|---|---|---|---|---|\n| 직접 작성 | | | | |\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 원문 대조용 보존본은 유지합니다. 현재 실습에는 아래 표의 수정 값을 적용하세요.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| G 척도 경계 | G 이하/이상 혼용; 규칙2=미션 2 값 | 보류G≥4 / 통과후보G≤3 및 독립 교차확인 일치 | 경계 G3 중첩 제거 | 미션 2 서식4·미션 4 규칙2·미션 5 분기2 |\n| 전문 검토 순서 | 분기1→2→3→4→5 | 번호 유지, 순서4→3→1→2→5 | 전문 검토를 위험·신뢰도보다 먼저 의뢰 | B3 기술검토 분기3, B5 법무검토 분기4; 승인 아님 |\n| 분할 입력 | 길이 제한 안내: 시작 문서7절 | 현재 수행 단계 안의 분할 입력 절차 | 없는 참조 제거 | 미션 2 AI 비교 |\n| 보고 시각·추정값 | 전일 보고 인용인데 D일 소모4·보유8 | 전일 출처 시각 + D일19시 계산용 추정기입 명시 | 실사 시각과 추정 계산 구분; 수치 불변 | 해솔AM-1은 현재량 확인 후 사용, G4 유지 |\n| 요청 수신처·요약행 | M-1 본부 요청 / 요약표4행 | Q2 지원팀 요청(후속상신 미확인) / Q1~Q6 6행 | 수신처·행 수 정합 | 미션 6 실습 자료 서식1 |\n| 치환 기록 중복 | 치환 자료 끝 문단2회 | 동일 지시1회 | 중복 제거 | 미션 10 치환 적용 기록 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '03_참고자료/20261019_보완정오표.md': '# 2026-10-19 보완 실습 적용 기준\n\n원문 계열 표·목표를 덮어쓰지 않고 별도 보완 실습본을 제공합니다. 아래 안내에 따라 대장과 용어를 해석하세요.\n\n| 항목 | 원문 값/종전 안내 | 수정 값/보완 안내 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 미션 9 재검증 | 원문 서식 2의 개선 전 판단은 제공표 값 | 원문 값 유지 + 별도 표에 내 이전 기준 재적용 판단·수정 기준 적용 판단 추가 | 과거 판단과 내 기준 개정 효과의 혼동 방지 | 미션 9 재검증 7건의 과정·근거; 원문 Q1 문장·제공표·미측정 과거 시간 불변 |\n| 미션 7 카드 범위 | 원문 목표·수행은 거점 교대 포함, T1~T6에는 기준선 포함·거점 교대 없음 | 원문 불변, 거점 교대는 제공 카드 학습 범위 밖이라고 명시 | 실제 카드로 수행한 범위만 보고 | 제공 T1~T6만 수행·채점; 추가 카드·평가 조건 없음 |\n| 미션 1 후보 검토 | AI 후보 전부 채택이면 보완 | 전체 후보의 근거 대조·유지 이유 없이 전부 채택하면 보완 | 폐기 0건 허용 원칙과 정합화 | 미션 1 수행 절차의 통과 기준·Q2와 파생 문서; 근거 있는 전부 채택 인정 |\n| 멈춘 분기의 뜻 | 아니오가 나와 보류로 빠지는 분기 번호 | 예/아니오와 관계없이 보류·추가 확인·전문 검토로 넘긴 지점의 분기 번호 | 예 경로 중단도 기록 가능해야 함 | 시작 용어표·미션 5·미션 10 기록 안내, 원문 트리 불변 |\n', '02_데이터셋/log04_m01_sample_2.md': "# LOG04 미션 1 실습 샘플\n\n> `02_데이터셋/LOG04_실습자료_수정실습본.md`의 미션 1 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 1. 지속 사용 환경에서의 핵심 관리 물자(주요 소모품, 드론 부품, 보호 자재) 소모율 분석**\n\n가. 제공 자료 1 - 거점별 가상 보유·소모 기록(D일 기준) [교육용가상자료]\n\n| **거점** | **품목** | **보유량** | **일평균 소모량** | **리드타임(일)** |\n| --- | --- | --- | --- | --- |\n| **북곡** | AM-1 | 24 | 6 | 3 |\n| **북곡** | DP-2 | 40 | 5 | 2 |\n| **북곡** | PG-3 | 120 | 10 | 1 |\n| **서림** | AM-1 | 18 | 3 | 3 |\n| **서림** | DP-2 | 12 | 4 | 2 |\n| **서림** | PG-3 | 30 | 15 | 1 |\n| **해솔** | AM-1 | 8 | 4 | 3 |\n| **해솔** | DP-2 | 20 | 2 | 2 |\n| **해솔** | PG-3 | 60 | 6 | 1 |\n\n가. 제공 자료 2 - 가상 소모 사례 5건 [교육용가상자료]\n\n| **사례** | **상황 요약** | **기록된 근거** | **눈에 띄는 한계** |\n| --- | --- | --- | --- |\n| **C1** | 북곡 거점 D-2일부터 AM-1 일 소모량이 3발에서 6발로 증가 | 거점 소모 일지 3일치 | 현장 대응 빈도 증가인지 손실인지 구분 기록 없음 |\n| **C2** | 서림 거점 DP-2가 이틀 만에 보유량의 절반이 소진됨 | 부품 불출 대장 | 파손 손실과 정상 소모가 한 칸에 합산됨 |\n| **C3** | 해솔 거점 PG-3는 여유가 있으나 추가 배정 요청이 올라옴 | 요청서 1장 | 요청 사유가 '예방'이라고만 적혀 있음 |\n| **C4** | 수송로 상태 불량으로 지원팀 도착 예정이 1일 늦어짐 | 수송 상황 보고 1건 | 출처가 단일 보고이고 복구 시점 미기재 |\n| **C5** | 'AM-1이 거의 소진되었다'는 구두 보고 | 없음 | 수량, 시각, 보고자 미기재 |\n\n나. 작성 서식 1 - 소모율·보유일수 계산표(제공 자료 1의 9개 행을 계산한다)\n\n| **거점** | **품목** | **소모율(%)** | **보유일수** | **리드타임 여유** | **먼저 눈에 띄는 점** |\n| --- | --- | --- | --- | --- | --- |\n| **북곡** | AM-1 | | | | |\n| **북곡** | DP-2 | | | | |\n| **북곡** | PG-3 | | | | |\n| **서림** | AM-1 | | | | |\n| **서림** | DP-2 | | | | |\n| **서림** | PG-3 | | | | |\n| **해솔** | AM-1 | | | | |\n| **해솔** | DP-2 | | | | |\n| **해솔** | PG-3 | | | | |\n\n나. 작성 서식 2 - 사례별 소모 급증 원인(사실과 추정 구분)\n\n| **사례** | **사실로 기록된 것** | **추정인 것** | **추가로 확인할 것** |\n| --- | --- | --- | --- |\n| **C1** | | | |\n| **C2** | | | |\n| **C3** | | | |\n| **C4** | | | |\n| **C5** | | | |\n\n나. 작성 서식 3 - 기준서 목차 골격(책임자 결정 의제 순으로 배열)\n\n| **순서** | **장 제목** | **담을 핵심 내용** | **책임자가 이 장에서 얻는 판단 정보** |\n| --- | --- | --- | --- |\n| **제1장** | | | |\n| **제2장** | | | |\n| **제3장** | | | |\n| **제4장** | | | |\n| **제5장** | | | |\n| **제6장** | | | |\n| **제7장** | | | |\n| **제8장** | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 가상 소모 사례 5건을 모두 검토하고 사실과 추정을 구분했다 | ( ) |\n| 9개 거점·품목의 소모율, 보유일수, 리드타임 여유를 계산했다(계산 오류 0건) | ( ) |\n| C5처럼 수량과 보고자가 없는 구두 보고를 찾아 표시했다 | ( ) |\n| 기준서 목차 8개 장의 제목과 핵심 내용을 정의했다 | ( ) |\n| 목차를 책임자가 먼저 알아야 할 의제 순으로 배열하고 이유를 한 줄로 적었다 | ( ) |\n", '02_데이터셋/LOG04_실습자료_수정실습본.md': "# LOG04 실습 자료\n\n[교육용가상자료] 원문 대조용 보존본과 분리한 현행 실습·채점 자료입니다. 03_참고자료/20261019b_정오표.md의 원문 값·수정 값·적용 범위를 확인합니다.\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\nG 척도는 숫자가 작을수록 신뢰가 높다. 통과 후보는 G1·G2·G3(숫자 코드 ≤3)이면서 독립 교차 확인 일치, 보류는 G4·G5·G6(숫자 코드 ≥4) 또는 교차 확인 불일치·미확인이다. 미션 2 서식 4의 통과 상한 G3과 미션 4 규칙 2의 보류 하한 G4를 구분한다.\n\n수정 트리의 평가 순서는 분기 4→3→1→2→5다(기존 분기 번호는 유지). 먼저 민간 물자 쟁점은 분기 4에서 법무 검토로 보류하고, 다음으로 대체품 전환 필요·기술 검토 미완료는 분기 3에서 기술 검토로 보류한다. 대체품 불필요는 해당 없음으로 통과한다. 그 뒤 분기 1 위험 등급, 분기 2 보고·교차 확인, 분기 5 승인권자 확인을 적용한다. 전문 검토 의뢰는 승인이나 정보 보완 완료가 아니다. 미확인 등급·수량을 추정해 채우지 않는다. 도달한 순서와 중단 분기, 미도달 분기를 각각 기록한다.\n\n초보 시간 계획: 원문 소요시간을 계획 기준으로 삼고 파일 열기·표 작성·AI 대기·재검토 시간을 별도로 확보한다. 초보자는 원문 시간보다 오래 걸릴 수 있다. 초보 소요시간은 원문의 약 1.3~1.7배를 준비 계획용 추정 범위로 참고하세요. 파일 조작·개념 학습·대기·재시도에 따라 달라지며 실제 학습자 실측이나 확정 시간이 아닙니다. 실제 시작·종료·중단 시간을 기록하고 카드·발표 제한시간은 별도로 판정한다.\n\n**LOG04 실습자료 - 드론·소모품 지속 사용 대비 핵심 관리 물자 재보급 우선순위 기준서**\n\n구 분 **실습자료 [교육용가상자료]**\n\n**1. 사용 안내**\n\n이 자료는 시나리오 미션 1~10을 수행할 때 바로 쓰는 가상 소모 기록, 가상 AAR 일지, 작성 서식, 점검표를 담고 있다. 모든 조직·물자·수량·시각은 [교육용가상자료]이며 실제와 관계없다.\n\n- 미션마다 '가. 제공 자료 → 나. 작성 서식 → 다. 점검표' 순서로 사용한다. 서식의 빈칸 '( )'과 빈 셀은 학습자가 채운다.\n\n- 개인 작업을 먼저 하고 조별(4~6명) 토의로 합의안을 만든다. 조별 역할은 조장(분석 책임), 기록자, 평가자, 레드팀으로 나누고 미션마다 돌아가며 맡는다.\n\n- 재보급 판단은 '승인', '보류 유지', '추가 확인 요청' 세 가지 중 하나로만 기록한다. 보고 자료의 신뢰도는 G1(가장 강함)~G6(근거 없음) 6단계로 구분한다.\n\n- 작성한 서식은 미션 10 기준서의 본문과 부록으로 다시 쓰이므로 버리지 말고 모아 둔다.\n\n- 마지막 '4. 평가 채점표'로 미션별 PASS/FAIL을 판정한다.\n\n| 안전 원칙: 본 자료의 등급 기준, 승인 권한 구분, 의사결정 트리는 우선순위 판단과 기록 절차를 연습하기 위한 학습용 가상 기준이다. 실제 시설 보호 자산 현황이나 물자 제원은 다루지 않는다. 재보급 우선순위의 최종 결정은 사람(책임자·승인권자)이 내리며 실제 시설 보호 현장 대응·보호 판단은 운영 규칙과 책임 체계에 따른다. 실제 조직 자료를 입력하거나 조직 내에서 촬영하지 않는다. |\n| --- |\n\n**2. 가상 상황 설정**\n\n| **가상 조직** | 가상 조직 A팀(보호 거점 3개), 상급 가상 지원조직 C팀, 배정 기관 가상 상급 지원본부 [교육용가상자료] |\n| --- | --- |\n\n| **가상 거점** | 가상 '북곡 거점', '서림 거점', '해솔 거점'. 실제 지형·위치와 관계없다. |\n| --- | --- |\n\n| **가상 핵심 관리 물자** | AM-1(가상 핵심 대응 장비, 단위 발), DP-2(가상 드론 부품 세트, 단위 세트), PG-3(가상 보호 자재 묶음, 단위 묶음). 제원은 없고 수량만 가상으로 쓴다. |\n| --- | --- |\n\n| **시간** | D일 기준 보유 현황, D+1~D+3일 가상 소모 시나리오. 재보급은 물자지원팀을 거쳐 리드타임(일)만큼 걸린다. |\n| --- | --- |\n\n| **핵심 계산** | 소모율(%) = 일평균 소모량 / 보유량 x 100, 보유일수 = 보유량 / 일평균 소모량, 리드타임 여유 = 보유일수 - 리드타임 |\n| --- | --- |\n\n| **제한** | 실제 재고·제원·수송로는 쓰지 않는다. 점수나 등급은 사람의 승인을 대신하지 않는다. 직책은 가상 직책(거점 물자담당, 팀 물자 담당자, 지원팀장, 본부 배정 담당, 기술 검토 담당, 법무 검토 담당)으로만 적는다. |\n| --- | --- |\n\n**3. 미션별 실습자료**\n\n**미션 1. 지속 사용 환경에서의 핵심 관리 물자(주요 소모품, 드론 부품, 보호 자재) 소모율 분석**\n\n가. 제공 자료 1 - 거점별 가상 보유·소모 기록(D일 기준) [교육용가상자료]\n\n| **거점** | **품목** | **보유량** | **일평균 소모량** | **리드타임(일)** |\n| --- | --- | --- | --- | --- |\n| **북곡** | AM-1 | 24 | 6 | 3 |\n| **북곡** | DP-2 | 40 | 5 | 2 |\n| **북곡** | PG-3 | 120 | 10 | 1 |\n| **서림** | AM-1 | 18 | 3 | 3 |\n| **서림** | DP-2 | 12 | 4 | 2 |\n| **서림** | PG-3 | 30 | 15 | 1 |\n| **해솔** | AM-1 | 8 | 4 | 3 |\n| **해솔** | DP-2 | 20 | 2 | 2 |\n| **해솔** | PG-3 | 60 | 6 | 1 |\n\n가. 제공 자료 2 - 가상 소모 사례 5건 [교육용가상자료]\n\n| **사례** | **상황 요약** | **기록된 근거** | **눈에 띄는 한계** |\n| --- | --- | --- | --- |\n| **C1** | 북곡 거점 D-2일부터 AM-1 일 소모량이 3발에서 6발로 증가 | 거점 소모 일지 3일치 | 현장 대응 빈도 증가인지 손실인지 구분 기록 없음 |\n| **C2** | 서림 거점 DP-2가 이틀 만에 보유량의 절반이 소진됨 | 부품 불출 대장 | 파손 손실과 정상 소모가 한 칸에 합산됨 |\n| **C3** | 해솔 거점 PG-3는 여유가 있으나 추가 배정 요청이 올라옴 | 요청서 1장 | 요청 사유가 '예방'이라고만 적혀 있음 |\n| **C4** | 수송로 상태 불량으로 지원팀 도착 예정이 1일 늦어짐 | 수송 상황 보고 1건 | 출처가 단일 보고이고 복구 시점 미기재 |\n| **C5** | 'AM-1이 거의 소진되었다'는 구두 보고 | 없음 | 수량, 시각, 보고자 미기재 |\n\n나. 작성 서식 1 - 소모율·보유일수 계산표(제공 자료 1의 9개 행을 계산한다)\n\n| **거점** | **품목** | **소모율(%)** | **보유일수** | **리드타임 여유** | **먼저 눈에 띄는 점** |\n| --- | --- | --- | --- | --- | --- |\n| **북곡** | AM-1 | | | | |\n| **북곡** | DP-2 | | | | |\n| **북곡** | PG-3 | | | | |\n| **서림** | AM-1 | | | | |\n| **서림** | DP-2 | | | | |\n| **서림** | PG-3 | | | | |\n| **해솔** | AM-1 | | | | |\n| **해솔** | DP-2 | | | | |\n| **해솔** | PG-3 | | | | |\n\n나. 작성 서식 2 - 사례별 소모 급증 원인(사실과 추정 구분)\n\n| **사례** | **사실로 기록된 것** | **추정인 것** | **추가로 확인할 것** |\n| --- | --- | --- | --- |\n| **C1** | | | |\n| **C2** | | | |\n| **C3** | | | |\n| **C4** | | | |\n| **C5** | | | |\n\n나. 작성 서식 3 - 기준서 목차 골격(책임자 결정 의제 순으로 배열)\n\n| **순서** | **장 제목** | **담을 핵심 내용** | **책임자가 이 장에서 얻는 판단 정보** |\n| --- | --- | --- | --- |\n| **제1장** | | | |\n| **제2장** | | | |\n| **제3장** | | | |\n| **제4장** | | | |\n| **제5장** | | | |\n| **제6장** | | | |\n| **제7장** | | | |\n| **제8장** | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 가상 소모 사례 5건을 모두 검토하고 사실과 추정을 구분했다 | ( ) |\n| 9개 거점·품목의 소모율, 보유일수, 리드타임 여유를 계산했다(계산 오류 0건) | ( ) |\n| C5처럼 수량과 보고자가 없는 구두 보고를 찾아 표시했다 | ( ) |\n| 기준서 목차 8개 장의 제목과 핵심 내용을 정의했다 | ( ) |\n| 목차를 책임자가 먼저 알아야 할 의제 순으로 배열하고 이유를 한 줄로 적었다 | ( ) |\n\n**미션 2. 조직 임무 중요도, 잔여 보유량, 재보급 리드타임 기반 우선순위 산정 기준 수립**\n\n가. 제공 자료 1 - 가상 재고 보고 5종 [교육용가상자료]\n\n| **보고** | **거점·품목** | **보고 내용** | **확인 방법** | **확인 상태** |\n| --- | --- | --- | --- | --- |\n| **R1** | 북곡 AM-1 | 보유 24, 소모 6/일 | 창고 실사 + 담당자·검증 담당 서명 | 실사표와 서명 모두 있음 |\n| **R2** | 서림 DP-2 | 보유 12, 소모 4/일 | 불출 대장 집계 | 집계는 있으나 실사 없음 |\n| **R3** | 해솔 AM-1 | 보유 8, 소모 4/일 | 전일 보고서 인용 | 전일 출처에 D일 소모를 추정 반영; 실사 미확인 |\n| **R4** | 서림 PG-3 | 보유 30 정도 | 담당자 구두 전달 | 수량이 어림값, 서명 없음 |\n| **R5** | 북곡 DP-2 | 보유 충분 | 근거 없음 | 수량·시각 없음 |\n\n가. 제공 자료 2 - 보고 신뢰도 6단계 틀. 단계 이름은 고정하고 판정 기준을 조별로 작성한다.\n\n| **단계** | **이름** | **판정 기준(조별 작성)** |\n| --- | --- | --- |\n| **G1** | 실사·독립 재확인 | |\n| **G2** | 실사 또는 대장 대조 | |\n| **G3** | 대장 집계 | |\n| **G4** | 전일 자료·인용 | |\n| **G5** | 구두·어림값 | |\n| **G6** | 근거 없음 | |\n\n나. 작성 서식 1 - 우선순위 산정 요소 척도표(구간은 관찰 가능한 수치로 쓴다)\n\n| **산정 요소** | **높음(점수 3)** | **보통(점수 2)** | **낮음(점수 1)** | **자료 출처** |\n| --- | --- | --- | --- | --- |\n| **임무 중요도** | | | | |\n| **보유일수** | | | | |\n| **리드타임 여유** | | | | |\n| **대체 가능성** | | | | |\n\n나. 작성 서식 2 - 평가 지표(10개 이상, 예시 2개 포함)\n\n| **번호** | **평가 지표** | **계산 방법(관찰 가능한 기준)** | **기록 책임자(가상 직책)** |\n| --- | --- | --- | --- |\n| **1** | 보유일수 | 보유량 / 일평균 소모량(소수 첫째 자리까지) | 거점 물자담당 |\n| **2** | 리드타임 여유 | 보유일수 - 리드타임(일). 음수이면 고갈 후 도착 | 팀 물자 담당자 |\n| **3** | | | |\n| **4** | | | |\n| **5** | | | |\n| **6** | | | |\n| **7** | | | |\n| **8** | | | |\n| **9** | | | |\n| **10** | | | |\n| **11** | | | |\n| **12** | | | |\n\n나. 작성 서식 3 - 등급 부여 일관성 시험(조원이 각자 G1~G6을 매기고 비교)\n\n| **보고** | **조원 1** | **조원 2** | **조원 3** | **조원 4** | **일치 여부** | **불일치 원인** |\n| --- | --- | --- | --- | --- | --- | --- |\n| **R1** | | | | | | |\n| **R2** | | | | | | |\n| **R3** | | | | | | |\n| **R4** | | | | | | |\n| **R5** | | | | | | |\n\n나. 작성 서식 4 - 교차 확인 필수 규칙: 단일 거점 보고는 G4 이상(숫자 코드 4·5·6)으로 보류하고 통과 후보는 G3 이하(숫자 코드 1·2·3)로 구분한다. 독립 교차 확인이 필요하며 긴급 재보급 승인은 서로 다른 출처(실사, 대장, 상급 집계) ( )건 이상이 일치할 때만 검토한다. 보고 시각이 ( )시간 이상 지난 자료는 ( )로 처리하고 재확인을 요청한다.\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 우선순위 산정 요소 4개의 척도를 수치 구간으로 적었다 | ( ) |\n| 신뢰도 G1~G6의 판정 기준을 모두 작성했다 | ( ) |\n| 평가 지표를 10개 이상 정의하고 계산 방법과 기록 책임자를 적었다 | ( ) |\n| 가상 재고 보고 5종에 대한 조원 등급이 4건 이상 일치했다(불일치 건은 원인과 수정 내용을 적었다) | ( ) |\n| 단일 출처 의존 경고와 교차 확인 필수 규칙을 명시했고 보고자 편향 방지 문구를 넣었다 | ( ) |\n\n**미션 3. 관리 물자 고갈 위험 등급 및 긴급 재보급 요청 승인/보류 기준서 초안 작성**\n\n가. 제공 자료 1 - 어긋난 가상 재고 기록 쌍 4건 [교육용가상자료]\n\n| **쌍** | **기록 A** | **기록 B** | **어긋난 지점** |\n| --- | --- | --- | --- |\n| **K1** | 서림 DP-2 보유 12세트(거점 일지, D일 18시) | 서림 DP-2 보유 20세트(지원팀 집계, D일 06시) | 시각 차이와 소모 반영 여부 |\n| **K2** | 북곡 PG-3 120묶음 | 북곡 PG-3 1,200개 | 단위 혼용(묶음/개) |\n| **K3** | 해솔 AM-1 8발(창고) | 해솔 AM-1 11발(창고+이동 중 3발) | 집계 범위 차이 |\n| **K4** | 서림 AM-1 소모 3/일(담당자 A) | 서림 AM-1 소모 5/일(담당자 B) | 보고자 차이, 기준 기간 미기재 |\n\n가. 제공 자료 2 - 고갈 위험 등급 틀(이름은 고정, 조건과 처리는 조별 작성)\n\n| **등급** | **조건(보유일수·리드타임 여유)** | **처리(승인·보류·추가 확인 요청)** | **책임 직책(가상)** |\n| --- | --- | --- | --- |\n| **정상** | | | |\n| **주의** | | | |\n| **경계** | | | |\n| **위험** | | | |\n\n나. 작성 서식 1 - 긴급 재보급 요청 승인/보류 기준서 초안\n\n| **승인 검토 조건(모두 충족)** | |\n| --- | --- |\n\n| **보류 조건(하나라도 해당)** | |\n| --- | --- |\n\n| **추가 확인 요청 조건** | |\n| --- | --- |\n\n| **기록해야 할 문서** | |\n| --- | --- |\n\n| **최종 결정자(가상 직책)** | |\n| --- | --- |\n\n나. 작성 서식 2 - 어긋난 기록 분류와 원인 후보\n\n| **쌍** | **분류(시점·단위·집계 범위·보고자)** | **원인 후보** | **확인 방법** | **책임 직책** |\n| --- | --- | --- | --- | --- |\n| **K1** | | | | |\n| **K2** | | | | |\n| **K3** | | | | |\n| **K4** | | | | |\n\n나. 작성 서식 3 - 자료 공백 요인과 불확실성 영향도(8개 이상)\n\n| **번호** | **공백 요인** | **영향도(높음·중간·낮음)** | **대체 확인 수단** |\n| --- | --- | --- | --- |\n| **1** | | | |\n| **2** | | | |\n| **3** | | | |\n| **4** | | | |\n| **5** | | | |\n| **6** | | | |\n| **7** | | | |\n| **8** | | | |\n| **9** | | | |\n\n나. 작성 서식 4 - 확신 표현 규칙: '충분하다'처럼 단정하는 표현 대신 '( ) 기준 보유일수 ( )일, 확인 단계 G( ), 확인되지 않은 것은 ( )'의 형식으로 쓴다.\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 위험 등급 4단계의 조건을 수치로 적고 등급별 처리를 연결했다 | ( ) |\n| 승인 검토·보류·추가 확인 조건을 구분했다 | ( ) |\n| 어긋난 기록 4건에 원인 후보와 확인 방법을 적었다 | ( ) |\n| 자료 공백 요인을 8개 이상 찾고 영향도와 대체 확인 수단을 적었다 | ( ) |\n| 판단의 한계를 함께 쓰는 표현 규칙을 정했다 | ( ) |\n\n**미션 4. 가상 현장 소모 시나리오 기반 물자별 재보급 우선순위 매트릭스 도출 실습**\n\n가. 제공 자료 1 - 가상 3일치 소모 시나리오(일 소모량) [교육용가상자료]\n\n| **거점·품목** | **D+1일** | **D+2일** | **D+3일** | **리드타임(일)** | **수송로 상태** |\n| --- | --- | --- | --- | --- | --- |\n| **해솔 AM-1(보유 8)** | 4 | 5 | 6 | 3 | 양호 |\n| **서림 DP-2(보유 12)** | 4 | 6 | 6 | 2 | 지연 1일 |\n| **서림 PG-3(보유 30)** | 15 | 15 | 12 | 1 | 양호 |\n\n가. 제공 자료 2 - 근거 항목 예시 4개(나머지는 조별로 12개 이상으로 늘린다)\n\n| **근거 항목** | **방향(상향·하향)** | **가중치 예시** |\n| --- | --- | --- |\n| **리드타임 여유가 0일 이하** | 상향 | +4 |\n| **소모량이 3일 연속 증가** | 상향 | +2 |\n| **보고 신뢰도가 G4 이상(숫자 코드 4·5·6; 신뢰 낮음)** | 하향 | -3 |\n| **대체품 전환이 이미 가능(기술 검토 완료)** | 하향 | -2 |\n\n나. 작성 서식 1 - 근거 항목 목록(12개 이상)\n\n| **번호** | **근거 항목** | **방향(상향·하향·확인 불가)** | **가중치** | **확인 방법** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n| **7** | | | | |\n| **8** | | | | |\n| **9** | | | | |\n| **10** | | | | |\n| **11** | | | | |\n| **12** | | | | |\n\n나. 작성 서식 2 - 종합 점수 산식과 규칙: 종합 점수 = (상향 근거 가중치 합) - (하향 근거 가중치 절댓값 합) 이다. 규칙 1 점수 ( )점 이상이면 긴급 요청 승인 검토, 규칙 2 점수가 높아도 보고 신뢰도가 G4 이상(숫자 코드 4·5·6; 신뢰 낮음)이면 ( ), 규칙 3 확인 불가 항목이 ( )개 이상이면 ( ), 규칙 4 점수는 ( )를 대신하지 않고 승인권자가 최종 결정한다.\n\n나. 작성 서식 3 - 사례 3종 적용(S1 해솔 AM-1, S2 서림 DP-2, S3 서림 PG-3)\n\n| **사례** | **상향 근거 합** | **하향 근거 합** | **종합 점수** | **분류(긴급 승인 검토·보류 후 재확인·일반 보급)** | **판단 근거** |\n| --- | --- | --- | --- | --- | --- |\n| **S1** | | | | | |\n| **S2** | | | | | |\n| **S3** | | | | | |\n\n나. 작성 서식 4 - 레드팀 반박 기록\n\n| **번호** | **레드팀 반박(예: 이 산식은 소모량이 큰 거점에만 유리하다)** | **작성 팀 대응** | **보완한 최소 증거 요건** |\n| --- | --- | --- | --- |\n| **1** | | | |\n| **2** | | | |\n| **3** | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 근거를 12개 이상 항목화하고 방향과 가중치를 적었다 | ( ) |\n| 점수 산식과 규칙 1~4의 빈칸을 모두 채웠다 | ( ) |\n| 사례 S1~S3의 점수와 분류를 계산해 기록했다 | ( ) |\n| 점수가 승인을 자동으로 결정하지 않고 사람이 최종 결정한다고 명시했다 | ( ) |\n| 레드팀 반박 3건에 대응했다 | ( ) |\n\n**미션 5. 물자 배분 불균형 해소를 위한 상급 지원본부 배정 원칙 및 대체품 전환 지침 수립**\n\n가. 제공 자료 1 - 가상 배정 상황과 경계선 사례 5종 [교육용가상자료]\n\n| **묶음** | **내용** | **애매한 이유** |\n| --- | --- | --- |\n| **B1** | 서림 DP-2 위험 등급, 보고 신뢰도 G3, 지원팀 비축분 일부 사용 요청 | 비축분 사용 조건이 불분명 |\n| **B2** | 해솔 AM-1 보유일수가 리드타임보다 짧으나 보고가 전일 자료(G4) | 위험은 크지만 자료가 약함 |\n| **B3** | 서림 PG-3 부족, 가상 대체품(기술 검토 미완료) 전환 제안 | 대체품 안전성 확인 전 |\n| **B4** | 북곡 거점이 '예방' 명목으로 DP-2 추가 배정 요청, 보유일수 8일 | 필요성이 낮은 요청 |\n| **B5** | 민간 지원 물자 일부를 보호 자재로 돌리자는 제안 | 군 물자가 아닌 물자, 법무 검토 필요 |\n\n가. 제공 자료 2 - 책임 직책 예시(가상): 거점 물자담당, 팀 물자 담당자, 지원팀장, 본부 배정 담당, 기술 검토 담당, 법무 검토 담당\n\n나. 작성 서식 1 - 의사결정 트리(조건과 분기를 채운다)\n\n| **단계** | **확인 질문** | **예일 때** | **아니오일 때** |\n| --- | --- | --- | --- |\n| **4 (먼저)** | 민간 지원 물자 등 군 물자가 아닌 쟁점이 있는가 | 법무 검토로 보류 | 분기 3 |\n| **3** | 대체품 전환이 필요하고 기술 검토가 미완료인가 | 기술 검토로 보류 | 불필요는 해당 없음으로 기록, 분기 1 |\n| **1** | 고갈 위험 등급이 경계 이상인가 | 분기 2 | 일반 보급 검토; 등급 미확인은 추가 확인 요청 |\n| **2** | 보고 신뢰도가 G3 이하(숫자 코드 1·2·3)이고 독립 교차 확인이 일치하는가 | 분기 5 | 보류 후 재확인 |\n| **5** | 승인 권한자의 확인이 있는가 | 사람 확인 기록 후 승인 | 보류 유지 |\n\n나. 작성 서식 2 - 배정 원칙과 대체품 전환 지침, 분기점 조건·책임 직책 매핑표\n\n| **분기점** | **조건(수치 또는 관찰 가능한 문장)** | **책임 직책(가상)** | **기록할 문서** |\n| --- | --- | --- | --- |\n| **T1** | | | |\n| **T2** | | | |\n| **T3** | | | |\n| **T4** | | | |\n\n나. 작성 서식 3 - 모의 결정 기록지(B1~B5)\n\n| **묶음** | **판단(승인·보류 유지·추가 확인 요청)** | **근거** | **책임 직책** | **조원 합의(예·아니오)** |\n| --- | --- | --- | --- | --- |\n| **B1** | | | | |\n| **B2** | | | | |\n| **B3** | | | | |\n| **B4** | | | | |\n| **B5** | | | | |\n\n나. 작성 서식 4 - 긴급 신속 승인 규칙: 신속 승인에서도 줄이지 않는 확인 단계는 ( ), 사후 감사 항목은 ( ), 사후 감사 기한은 ( )이다. 민간 지원 물자와 기술 검토 미완료 대체품은 신속 승인 대상에서 ( ).\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 분기점 4개의 조건을 수치 또는 관찰 가능한 문장으로 적었다 | ( ) |\n| 분기점마다 책임 직책(가상)과 기록 문서를 연결했다 | ( ) |\n| 경계선 사례 5종 모두에 판단과 근거를 기록했다 | ( ) |\n| 기술 검토 전 대체품 전환과 민간 지원 물자 전용이 막히고 법무·기술 검토로 연결된다 | ( ) |\n| 사람의 승인 없이 재보급이 승인되는 경로가 없고 운영 규칙·책임 체계 우선 문구가 있다 | ( ) |\n\n**미션 6. 책임자 결정용 핵심 관리 물자 재보급 현황 및 우선순위 브리핑 양식 작성**\n\n가. 제공 자료 1 - 혼동 유발 표현 예시(나머지는 조별로 6개 이상으로 늘린다)\n\n| **혼동 유발 표현** | **왜 혼동되는가** | **대체 표현 예** |\n| --- | --- | --- |\n| **충분하다** | 기준과 수량이 보이지 않음 | 보유 (수량), 보유일수 (일), 리드타임 여유 (일) |\n| **곧 도착한다** | 도착 시각이 없음 | 지원팀 출발 (시각), 도착 예정 (시각), 확인 단계 G( ) |\n| **거의 소진** | 수치 없는 어림 | 보유 (수량)이 일평균 소모 (수량)의 ( )일치 |\n\n가. 제공 자료 2 - 가상 현황 메모 3종(사실·판단·조치·미확인을 구분해 표시한다)\n\n| **메모** | **내용** |\n| --- | --- |\n| **M-1** | 서림 DP-2는 불출 대장상 보유 12세트, 일 소모 4세트다. 곧 바닥난다고 본다. 지원팀에 긴급 요청(Q2)을 올렸다(본부 후속 상신 여부는 미확인). 파손 손실 포함 여부는 모른다. |\n| **M-2** | 해솔 PG-3는 충분하다고 판단된다. 추가 배정은 하지 않기로 했다. 야간 소모량은 아직 반영되지 않았다. |\n| **M-3** | 북곡 AM-1 소모가 늘었다. 현장 대응 빈도 때문이라고 생각한다. 다음 보고에 일지를 첨부하라고 지시했다. 손실 발생 여부는 확인되지 않았다. |\n\n나. 작성 서식 1 - 책임자 보고용 재보급 현황 요약 차트(Q1~Q6, 6행)\n\n| **요청ID** | **거점** | **품목** | **보유일수** | **리드타임 여유** | **위험 등급** | **요청 수량** | **결정 요청** |\n|---|---|---|---|---|---|---|---|\n| Q1 | | | | | | | |\n| Q2 | | | | | | | |\n| Q3 | | | | | | | |\n| Q4 | | | | | | | |\n| Q5 | | | | | | | |\n| Q6 | | | | | | | |\n\n나. 작성 서식 2 - 기록 분류 규칙(빈칸을 채운다): 사실은 ( ), 판단은 ( ), 조치는 ( ), 미확인은 ( ) 표시로 구분한다. 문장 하나에는 한 가지 분류만 담는다.\n\n나. 작성 서식 3 - 요약형 기록(메모 M-1~M-3을 옮겨 적는다)\n\n| **메모** | **사실** | **판단** | **조치** | **미확인** |\n| --- | --- | --- | --- | --- |\n| **M-1** | | | | |\n| **M-2** | | | | |\n| **M-3** | | | | |\n\n나. 작성 서식 4 - 상세형 기록(1건 선택)\n\n| **기록 번호** | |\n| --- | --- |\n\n| **사실(거점, 품목, 수량, 시각, 신뢰도 단계)** | |\n| --- | --- |\n\n| **판단(위험 등급과 근거)** | |\n| --- | --- |\n\n| **조치(누가, 언제까지)** | |\n| --- | --- |\n\n| **미확인(확인 방법, 책임 직책)** | |\n| --- | --- |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 요약 차트에 보유일수, 리드타임 여유, 위험 등급, 결정 요청 칸이 있다 | ( ) |\n| 혼동 유발 표현 금지 목록을 6개 이상 만들고 대체 표현을 적었다 | ( ) |\n| 메모 3종에서 사실과 판단을 혼동한 문장이 0건이다 | ( ) |\n| 요약형과 상세형이 같은 내용을 담고 있다 | ( ) |\n| 미확인 항목마다 확인 방법과 책임 직책이 있다 | ( ) |\n\n**미션 7. 수송로 차단 및 물자 80% 고갈 모의 재보급 의사결정 스트레스 훈련**\n\n가. 제공 자료 1 - 스트레스 모의 카드 T1~T6 [교육용가상자료] (운영 담당자가 순서대로 주입)\n\n| **카드** | **주입 내용** | **확인할 것** |\n| --- | --- | --- |\n| **T1** | 기준선: 3개 거점의 D일 보유·소모 현황 | 카드별 우선순위 기준선 기록 |\n| **T2** | 수송로 차단: 지원팀 도착이 3일 늦어짐 | 리드타임 여유 재계산 |\n| **T3** | 물자 80% 소진: AM-1 보유가 거점 평균 20% 수준으로 감소 | 위험 등급 재분류와 승인·보류 |\n| **T4** | 보고 지연: 서림 거점 보고가 12시간 늦음 | 신뢰도 단계 하향과 재확인 요청 |\n| **T5** | 대체 보급로 후보 등장(상태 미확인) | 승인 전 확인할 사항 |\n| **T6** | 추가 요청 폭주: 3개 거점이 모두 긴급 요청 | 배정 원칙 적용과 결정 요청 |\n\n가. 제공 자료 2 - 루브릭 틀(5영역 x 4등급, 관찰 가능한 행동으로 작성)\n\n| **평가 영역** | **탁월** | **우수** | **보통** | **미흡** |\n| --- | --- | --- | --- | --- |\n| **소모 파악** | | | | |\n| **우선순위 판단** | | | | |\n| **승인·보류 근거** | | | | |\n| **대체 방안** | | | | |\n| **보고 적시성** | | | | |\n\n나. 작성 서식 1 - 카드별 재산정 기록\n\n| **카드** | **재산정 결과(거점·품목 순위)** | **승인·보류** | **근거** | **소요 시간(분)** |\n| --- | --- | --- | --- | --- |\n| **T1** | | | | |\n| **T2** | | | | |\n| **T3** | | | | |\n| **T4** | | | | |\n| **T5** | | | | |\n| **T6** | | | | |\n\n나. 작성 서식 2 - 평가자 채점 일치표(평가자 2명이 상의 없이 채점)\n\n| **평가 영역** | **평가자 1 등급** | **평가자 2 등급** | **일치(예·아니오)** | **불일치 이유** |\n| --- | --- | --- | --- | --- |\n| **소모 파악** | | | | |\n| **우선순위 판단** | | | | |\n| **승인·보류 근거** | | | | |\n| **대체 방안** | | | | |\n| **보고 적시성** | | | | |\n\n나. 작성 서식 3 - 판단이 흔들린 지점: 순위가 가장 크게 바뀐 카드는 ( ), 이유는 ( ), 이 지점에서 책임자에게 알릴 문장은 '( )'이다. 일치율은 ( ) / 5 = ( )%이다.\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 5개 영역 x 4단계 기준을 모두 관찰 가능한 행동으로 정의했다 | ( ) |\n| 카드 T1~T6마다 우선순위 재산정과 승인·보류를 기록했다 | ( ) |\n| 카드당 재산정 시간이 10분 이내이거나 지연 사유를 적었다 | ( ) |\n| 평가자 2명이 상의 없이 채점하고 일치율을 계산했다 | ( ) |\n| 물자 부족 상황에서도 사람의 승인과 신뢰도 확인 단계를 생략하지 않았다 | ( ) |\n\n**미션 8. AAR 회의를 통한 비효율적 물자 할당 사례 분석 및 우선순위 기준 보정**\n\n가. 제공 자료 - 가상 AAR 일지 A·B [교육용가상자료]\n\n| **시각** | **AAR 일지 A (과잉 배정 사례)** | **AAR 일지 B (자료 지연 부족 사례)** |\n| --- | --- | --- |\n| **D+1 08:00** | 해솔 PG-3 추가 배정 요청 접수(보유일수 10일) | 서림 DP-2 보고 도착, 전일 18시 자료 |\n| **D+1 10:30** | 요청 사유 '예방', 신뢰도 확인 없이 승인 | 지원팀 비축 확인, 서림 요청은 일반 보급으로 분류 |\n| **D+1 15:00** | 해솔에 PG-3 60묶음 추가 도착 | 서림 DP-2 소모 급증, 보유 4세트로 감소 |\n| **D+2 07:00** | 서림 PG-3 보유 6묶음, 긴급 요청 발생 | 서림 긴급 요청, 리드타임 2일 |\n| **D+2 12:00** | 북곡에서 서림으로 물자 이동 검토, 수송 불가 | D+3 도착 예정, 그동안 서림 DP-2 보유 0세트 |\n\n나. 작성 서식 1 - 계획과 실제 차이, 원인 분류(사람·절차·정보·환경, 개인 비난 표현 금지)\n\n| **번호** | **계획** | **실제** | **원인 분류** | **원인 설명(사실 중심)** | **개선안** |\n| --- | --- | --- | --- | --- | --- |\n| **1** | | | | | |\n| **2** | | | | | |\n| **3** | | | | | |\n| **4** | | | | | |\n| **5** | | | | | |\n\n나. 작성 서식 2 - 레드팀 쟁점·대응(5건 이상)\n\n| **번호** | **기준서의 취약한 전제** | **레드팀 비판** | **작성 팀 반박 또는 수정안** | **경고 문구** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n\n나. 작성 서식 3 - 반대 방향 위험 점검\n\n| **위험** | **설명** | **대응** |\n| --- | --- | --- |\n| **한 거점에 몰아주기 위험** | | |\n| **예비 물자 소진 위험** | | |\n| **기준 복잡도 증가로 현장 판단 지연** | | |\n\n나. 작성 서식 4 - 수정 반영 계획\n\n| **번호** | **수정할 문서와 위치** | **수정 내용** | **책임 직책(가상)** | **기한** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 일지 A·B의 계획과 실제 차이를 5건 이상 적었다 | ( ) |\n| 원인을 사람·절차·정보·환경으로 분류하고 개인 비난 표현이 없다 | ( ) |\n| 레드팀 쟁점 5건 이상에 반박 또는 수정안을 100% 적었다 | ( ) |\n| 취약한 전제에 경고 문구를 명시했다 | ( ) |\n| 반대 방향 위험 3가지를 점검했다 | ( ) |\n\n**미션 9. 피드백 반영 관리 물자 재보급 우선순위 기준서 수정 및 정합성 검증**\n\n가. 제공 자료 - 가상 재검증 사례 7건 [교육용가상자료]\n\n| **사례** | **내용** | **개선 전 판단** | **개선 후 판단(조별 작성)** |\n| --- | --- | --- | --- |\n| **V1** | 해솔 AM-1 위험 등급, 보고는 전일 자료(G4) | 승인 | |\n| **V2** | 서림 DP-2 소모 급증, 파손과 정상 소모가 합산됨 | 승인 | |\n| **V3** | 북곡 DP-2 '예방' 명목 요청, 보유일수 8일 | 승인 | |\n| **V4** | 서림 PG-3 경계 등급, 교차 확인 일치 | 보류 | |\n| **V5** | 기술 검토 미완료 대체품 전환 제안 | 승인 | |\n| **V6** | 수송 지연 보고가 단일 출처 | 승인 | |\n| **V7** | 민간 지원 물자 전용 제안 | 승인 | |\n\n나. 작성 서식 1 - 개선 전후 기준 비교표\n\n| **항목** | **개선 전 기준** | **개선 후 기준** | **바뀐 이유** | **근거 미션** |\n| --- | --- | --- | --- | --- |\n| **보고 신뢰도 단계** | | | | |\n| **교차 확인** | | | | |\n| **위험 등급 조건** | | | | |\n| **대체품 전환 조건** | | | | |\n| **민간 물자 점검** | | | | |\n\n나. 작성 서식 2 - 재검증 결과표\n\n| **사례** | **개선 전 판단** | **개선 후 판단** | **일치 여부** | **소요 시간(분)** |\n| --- | --- | --- | --- | --- |\n| **V1** | | | | |\n| **V2** | | | | |\n| **V3** | | | | |\n| **V4** | | | | |\n| **V5** | | | | |\n| **V6** | | | | |\n| **V7** | | | | |\n\n나. 작성 서식 3 - 1페이지 브리핑 BLUF 틀\n\n| **결론(첫 문장)** | |\n| --- | --- |\n\n| **핵심 결정 사항** | |\n| --- | --- |\n\n| **위험 요소(신호등 표시: 녹·황·적)** | |\n| --- | --- |\n\n| **대안 3종(장단점 한 줄)** | |\n| --- | --- |\n\n| **근거(미션·가상 자료)** | |\n| --- | --- |\n\n나. 작성 서식 4 - 브리핑 리허설 기록\n\n| **항목** | **목표** | **결과** |\n| --- | --- | --- |\n| 발표 시간 | 3분 이내 | |\n| 분량 | A4 1페이지 | |\n| 질의 응답 | 3건 이상 답변 | |\n| 결론 먼저 제시 | 첫 문장 | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 개선 전후 기준의 바뀐 조건과 이유를 모두 적었다 | ( ) |\n| 재검증 7건 모두 개선 전후 판단을 비교했다 | ( ) |\n| 브리핑이 결론부터 시작하고 A4 1페이지에 들어간다 | ( ) |\n| 핵심 결정 사항, 위험 요소, 대안 3종이 모두 있다 | ( ) |\n| 3분 안에 발표하고 질의에 답했다 | ( ) |\n| 기준 변경으로 생긴 판단 차이를 기준서 조항과 연결했다 | ( ) |\n\n**미션 10. 핵심 관리 물자 재보급 우선순위 결정 기준서 최종 표준안 완성**\n\n가. 제공 자료 1 - 지침서 8개 장 구성과 가져올 결과물\n\n| **장** | **제목** | **포함 내용** | **가져올 결과물** |\n| --- | --- | --- | --- |\n| **1** | 목적과 적용 범위 | 목적, 적용 상황, 적용하지 않는 상황, 운영 규칙과의 관계 | 미션 1 |\n| **2** | 용어 정의 | 소모율, 보유일수, 리드타임 여유, 위험 등급, 보류 | 미션 1~미션 3 |\n| **3** | 소모율 분석 | 계산 방법, 사례 분석 방법, 자료 공백 | 미션 1·미션 3 |\n| **4** | 우선순위 산정과 자료 신뢰도 | 산정 요소 척도, G1~G6, 평가 지표, 교차 확인 규칙 | 미션 2·미션 4 |\n| **5** | 위험 등급과 승인·보류 | 등급 조건, 승인·보류 기준, 의사결정 트리, 책임 직책 | 미션 3·미션 5 |\n| **6** | 배정 원칙과 대체품 전환 | 배정 원칙, 대체품 전환 지침, 긴급 신속 승인과 사후 감사 | 미션 5 |\n| **7** | 보고·훈련·평가 | 브리핑 양식, 스트레스 훈련, 루브릭, AAR, 레드팀 | 미션 6~미션 9 |\n| **8** | 개정 관리 | 연간 개정 주기, 의견 수렴 창구, 책임 직책, 이력 | 미션 8·미션 9 |\n\n가. 제공 자료 2 - 모의 심의 역할 카드\n\n| **역할** | **검토 질문** |\n| --- | --- |\n| **절차 역할** | 산정 요소와 신뢰도 단계가 일관되는가? 신병도 이해할 용어인가? |\n| **법무 역할** | '최종 결정은 사람이, 운영 규칙·책임 체계에 따라'가 드러나는가? 민간 지원 물자 쟁점이 법무 검토로 연계되는가? |\n| **현장 운영 역할** | 현장에서 20분 안에 판단할 수 있는가? 점수 계산 때문에 긴급 대응이 늦어지지 않는가? |\n\n나. 작성 서식 1 - 근거 표기 점검표(지침서 조항마다 작성)\n\n| **조항** | **기준 내용 요약** | **근거(미션·가상 자료)** | **표기 여부** |\n| --- | --- | --- | --- |\n| **1-1** | | | |\n| **2-1** | | | |\n| **3-1** | | | |\n| **4-1** | | | |\n| **5-1** | | | |\n| **6-1** | | | |\n| **7-1** | | | |\n| **8-1** | | | |\n\n나. 작성 서식 2 - 모의 심의 지적 사항 반영표\n\n| **번호** | **역할** | **지적 사항** | **반영 내용** | **반영 위치(장·조)** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n\n나. 작성 서식 3 - 개정 관리: 개정 주기 ( ), 의견 수렴 창구와 방법 ( ), 개정 책임 직책(가상) ( ), 개정 이력 기록 위치 ( )\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 8개 장과 부록을 모두 완성했다 | ( ) |\n| 모든 기준 조항에 근거(미션·가상 자료)를 표기했다 | ( ) |\n| 절차·법무·현장 운영 역할의 지적 사항을 반영하고 반영 위치를 적었다 | ( ) |\n| '최종 결정은 사람이 하며 실제 시설 보호 현장 대응 판단은 운영 규칙·책임 체계에 따른다'는 문구가 1장과 5장에 있다 | ( ) |\n| 대체품 전환은 기술 검토 후에만, 민간 지원 물자는 법무 검토 후에만 가능하다는 문구가 6장에 있다 | ( ) |\n| 연간 개정 주기와 실무 의견 수렴 창구를 정했다 | ( ) |\n| 실제 조직·재고·제원을 쓰지 않았다 | ( ) |\n\n**4. 평가 채점표**\n\n미션별 평가 기준은 시나리오의 평가 항목과 같다. 평가자는 기준 충족 여부를 PASS/FAIL로 표시하고 FAIL이면 사유를 비고에 적는다.\n\n| **미션** | **구분** | **채점 기준** | **판정** | **비고** |\n| --- | --- | --- | --- | --- |\n| **미션 1** | 필수 | 소모율 분석·문서 골격: 가상 사례·문헌 5건 이상을 검토했는가, 3개 품목의 소모율과 보유일수를 계산해 오류 없이 표로 정리했는가, 기준서 목차(8개 장) 골격을 100% 정의하고 책임자 결정 의제 순으로 배열했는가. | PASS / FAIL | |\n| **미션 2** | 필수 | 산정 기준·신뢰도 6단계: 신뢰도 평가 기준 6단계를 완비하고 평가 지표 10개 이상을 정의했는가, 가상 재고 보고 5종에 대한 조원 간 등급 일치가 4건 이상인가, 단일 출처 의존 경고와 교차 확인 필수 규칙을 명시했는가. | PASS / FAIL | |\n| **미션 3** | 필수 | 위험 등급·불확실성 평가: 관측 공백(자료 공백) 요인 8개 이상을 식별하고 영향도 매트릭스를 도출했는가, 어긋난 기록 4건에 원인 후보와 확인 방법을 적고 4단계 위험 등급의 승인/보류 처리를 명시했는가, 판단 한계와 대체 확인 수단을 적었는가. | PASS / FAIL | |\n| **미션 4** | 선택 | 근거 항목·우선순위 산식: 우선순위 판단 근거를 12개 이상 항목화하고 가중치 산식을 100% 정립했는가, 사례 3종의 점수와 분류를 기록했는가, 레드팀 반박에 대응해 점수만으로 단정하지 않는 최소 증거 요건을 설정했는가. | PASS / FAIL | |\n| **미션 5** | 필수 | 분기점·책임 매핑·모의 승인: 승인/보류 조건 분기점(Threshold)을 100% 명시하고 책임자 직책 매핑을 완료했는가, 경계선 사례 5종 모두에 판단과 근거를 기록했는가, 운영 규칙·책임 체계 우선 원칙과 사람의 최종 결정을 지키고 긴급 신속 승인과 사후 감사 규정을 보완했는가. | PASS / FAIL | |\n| **미션 6** | 선택 | 기록 분류·브리핑 서식: 기록 분류 규칙을 100% 표준화하고 혼동 유발 표현 금지 가이드를 6개 이상 포함했는가, 가상 현황 메모 3종에서 사실과 판단의 혼동이 0건인가, 요약형·상세형 듀얼 템플릿을 구성했는가. | PASS / FAIL | |\n| **미션 7** | 필수 | 스트레스 훈련·루브릭: 평가 영역 5개와 등급 4단계(탁월·우수·보통·미흡) 기준 정의율 100%인가, 카드 6장 모두에 재산정 기록이 있고 카드당 재산정 10분 이내인가, 평가자 2명 채점 일치율 80% 이상인가, 판단이 흔들린 지점을 기록했는가. | PASS / FAIL | |\n| **미션 8** | 선택 | 원인 규명·기준 보정: 도출된 비판적 쟁점 5건 이상에 대해 반박 및 수정 대안을 100% 수립했는가, 취약한 전제(Assumptions)에 경고 문구를 명시했는가, 반대 방향 위험을 점검하고 기준 보정 계획에 반영했는가. | PASS / FAIL | |\n| **미션 9** | 필수 | 기준서 수정·1페이지 브리핑: 핵심 결정 사항, 위험 요소, 대안 3종이 1페이지(A4) 안에 완결되는가, 재검증 7건의 개선 전후 비교와 정합성 검증을 마쳤는가, BLUF 준수(결론 먼저, 3분 이내 발표)와 신호등 표시가 있는가. | PASS / FAIL | |\n| **미션 10** | 필수 | 표준 지침서 통합 완성: 표준 지침서 총 8개 장 및 부록을 100% 완성하고 모든 기준 조항에 근거(미션·가상 자료) 출처를 표기했는가, 모의 심의 지적 사항을 반영하고 연간 개정 주기와 실무 의견 수렴 창구를 두었는가, '최종 결정은 사람이 하며 실제 현장 대응 판단은 운영 규칙·책임 체계에 따른다'는 문구를 명시했는가. | PASS / FAIL | |\n\n| **판정 기준** | 필수 7개 미션(미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10)이 모두 PASS여야 최종 PASS로 인정한다. 선택 미션(미션 4·미션 6·미션 8)은 가점·보완 의견에 반영한다. |\n| --- | --- |\n\n| **공통 감점** | 실제 조직명·재고·제원·실존 인물 정보를 사용한 경우, 조직 자료를 외부 AI 서비스에 입력한 경우 해당 미션은 FAIL로 처리한다. |\n| --- | --- |\n\n| **주제 안전 기준** | 단일 보고만으로 긴급 재보급을 승인하는 판단이 있거나 기준서에 사람의 최종 결정·운영 규칙 우선 문구가 없으면 미션 5·미션 10은 FAIL로 처리한다. |\n| --- | --- |\n\n| **최종 판정 결과** | ( PASS / FAIL ) 평가자 서명: ( ) |\n| --- | --- |\n\n| 안전 원칙: 본 자료의 등급 기준, 승인 권한 구분, 의사결정 트리는 학습용 가상 기준이다. 재보급 우선순위의 최종 결정은 사람(책임자·승인권자)이 내리며 실제 시설 보호 현장 대응·보호 판단은 운영 규칙과 책임 체계에 따른다. 실제 조직 자료를 입력하거나 조직 내에서 촬영하지 않는다. |\n| --- |\n", 'docs/log04_doc_framework.md': '# 미션 1 docs/log04_doc_framework.md 및 분석 개요서(품목별 소모율·보유일수 계산표, 사례 검토표, 기준서 목차 골격) (필수)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}
for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 1 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')


### 할 일

1. 거점·품목·수량·기간·단위를 원문에서 확인합니다.

2. AI에게 현황 분석표와 8개 장 기준서 목차의 초안을 요청합니다.

3. 소모율·보유일수·도착까지의 여유를 계산기로 계산하고 검산 열에 적습니다.

4. 사례별 사실·추정·확인할 것을 나누고 숫자가 다른 곳을 원자료와 대조합니다.

5. 계산표·분석 개요서·목차를 묶어 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

**핵심 수행·평가 포인트**
- 대장(`02_데이터셋/거점_보유소모_대장.csv`) 열 이름 14개를 적고, 9행 수량이 대장과 같은지 확인합니다.
- 소모율(%)·보유일수·리드타임 여유는 계산기/엑셀(또는 아래 `[보조 도구]` 셀)로 **소수 첫째 자리**까지 계산하며 **음수 여유는 음수 그대로** 적습니다.
- 검산 열(`D-3일보유량 − (D-2+D-1+D일소모) + 입고량 = D일보유량_보고`)을 9행 전부 확인하고, 불일치 행은 거점·품목과 열 이름을 밝혀 `'확인 후 사용'`으로 분류합니다.
- 사례 C1~C5는 요약하지 말고 원문 그대로 전사하며(C5 `'근거 없음'`), 사실(기록된 근거)과 추정(눈에 띄는 한계)을 구분합니다.
- 도전 과제: 해솔 AM-1 보유 8발(일평균 6발) vs 4발일 때 보유일수·여유를 소수 첫째 자리로 비교합니다.


### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「물자 현황 분석표와 기준서 목차」의 초안을 만들어 줘.
자료: 거점 현황 9행, C1~C5, 보유·소모 대장의 관련 행.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 거점 | 품목 | 수량 | 단위 | 일일 소모 | 보유일수 | 리드타임 여유 | 검산. 사례 | 사실 | 추정 | 확인할 것.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식

더블클릭해 빈칸을 채우고 `Shift+Enter`로 표시하세요. 별도 산출물 파일에도 옮겨 저장할 수 있습니다.

**미션 1. 물자 현황 분석표와 기준서 목차 (필수)**

나. 작성 서식 1 - 소모율·보유일수 계산표(제공 자료 1의 9개 행을 계산한다)

| **거점** | **품목** | **소모율(%)** | **보유일수** | **리드타임 여유** | **먼저 눈에 띄는 점** |
| --- | --- | --- | --- | --- | --- |
| **북곡** | AM-1 | | | | |
| **북곡** | DP-2 | | | | |
| **북곡** | PG-3 | | | | |
| **서림** | AM-1 | | | | |
| **서림** | DP-2 | | | | |
| **서림** | PG-3 | | | | |
| **해솔** | AM-1 | | | | |
| **해솔** | DP-2 | | | | |
| **해솔** | PG-3 | | | | |

나. 작성 서식 2 - 사례별 소모 급증 원인(사실과 추정 구분)

| **사례** | **사실로 기록된 것** | **추정인 것** | **추가로 확인할 것** |
| --- | --- | --- | --- |
| **C1** | | | |
| **C2** | | | |
| **C3** | | | |
| **C4** | | | |
| **C5** | | | |

나. 작성 서식 3 - 기준서 목차 골격(책임자 결정 의제 순으로 배열)

| **순서** | **장 제목** | **담을 핵심 내용** | **책임자가 이 장에서 얻는 판단 정보** |
| --- | --- | --- | --- |
| **제1장** | | | |
| **제2장** | | | |
| **제3장** | | | |
| **제4장** | | | |
| **제5장** | | | |
| **제6장** | | | |
| **제7장** | | | |
| **제8장** | | | |

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| 가상 소모 사례 5건을 모두 검토하고 사실과 추정을 구분했다 | ( ) |
| 9개 거점·품목의 소모율, 보유일수, 리드타임 여유를 계산했다(계산 오류 0건) | ( ) |
| C5처럼 수량과 보고자가 없는 구두 보고를 찾아 표시했다 | ( ) |
| 기준서 목차 8개 장의 제목과 핵심 내용을 정의했다 | ( ) |
| 목차를 책임자가 먼저 알아야 할 의제 순으로 배열하고 이유를 한 줄로 적었다 | ( ) |

> **보완 열 안내**: 작성 서식 1에 **검산 열**(`D-3일보유량 − (D-2+D-1+D일소모) + 입고량 = D일보유량_보고`)을 추가해 9행 전부를 확인하고, 불일치 행은 `'확인 후 사용'`으로 표시하세요.


### AI 대조·수정 기록 및 남길 기록

| 자료 ID | 초안 문장 | 고친 문장 | 고친 이유 |
|---|---|---|---|
| | | | |

**남길 기록**

1. 프롬프트 전문과 실제 응답(연속 5줄 이상, 짧으면 전문), 세션 구분(협업 시 역할·요청·독립 판단 원문)

2. 사실·숫자 대조 / 빠진 항목 / 과장 표현 확인

3. 전체 후보 대조 및 채택·폐기(유지) 이유

4. 점검 결과와 미수행 항목


In [ ]:
# [보조 도구] 미션 1 — 대장 9행 소모율·보유일수·여유·3일 검산 및 도전 과제 확인
import csv
from pathlib import Path

ledger = Path("02_데이터셋/거점_보유소모_대장.csv")
with ledger.open(encoding="utf-8") as f:
    next(f)  # 첫 줄 안내문 건너뛰기
    rows = list(csv.DictReader(f))

print(f"{'거점':<4} {'품목':<5} {'보유':>4} {'일소모':>5} {'리드':>4} | {'소모율(%)':>8} {'보유일수':>7} {'여유':>6} | {'검산기대':>7} {'보고':>4} {'일치':<6}")
print("-" * 82)
for r in rows:
    stock = float(r["D일보유량_보고"])
    daily = float(r["일평균소모량_보고"])
    lead = float(r["리드타임_일"])
    rate = round(daily / stock * 100, 1)
    days = round(stock / daily, 1)
    margin = round(days - lead, 1)
    expected = float(r["D-3일보유량"]) - (float(r["D-2일소모"]) + float(r["D-1일소모"]) + float(r["D일소모"])) + float(r["입고량_D-2~D"])
    ok = "일치" if expected == stock else "확인후사용"
    print(f"{r['거점']:<4} {r['품목']:<5} {stock:>4.0f} {daily:>5.0f} {lead:>4.0f} | {rate:>8.1f} {days:>7.1f} {margin:>6.1f} | {expected:>7.1f} {stock:>4.0f} {ok:<6}")

print("\n[도전 과제] 해솔 AM-1 (일평균 6발, 리드타임 3일):")
for qty in (8, 4):
    d = round(qty / 6, 1)
    m = round(d - 3, 1)
    print(f"  보유 {qty}발 -> 소모율 {round(6/qty*100, 1)}%, 보유일수 {d}일, 리드타임 여유 {m}일")


## [교수자용 해설 및 모범 답안 — 미션 1]

### 1. 지도 포인트 (평가 핵심 및 오답 함정)
- **소수 첫째 자리·음수 유지**: `해솔 AM-1`의 리드타임 여유 `-1.0`일을 `0`으로 바꾸거나 소모율(`8.3%`, `16.7%`, `33.3%`)을 정수로 반올림하면 감점/보완 대상입니다.
- **3일 검산 불일치 (`북곡 DP-2`)**: `D-3일보유량(60) − (5+5+5) + 0 = 45`인데 `D일보유량_보고`가 `40`입니다. 어느 쪽이 맞는지 임의로 단정하지 말고 **`북곡 DP-2, D일보유량_보고 열 — 확인 후 사용`**으로 분류해야 합니다.
- **C1~C5 전사 & C5 표시**: C1~C5는 요약하지 않고 원문 그대로 옮기고, C5에는 `'근거 없음'`을 표시합니다. AI 후보를 전부 채택하더라도 전체 후보 대조와 유지 이유가 있으면 충족합니다(`20261019_보완정오표.md`).

### 2. 모범 답안 — 작성 서식 1 (9행 계산·검산 요약)

| 거점 | 품목 | 소모율(%) | 보유일수 | 리드타임 여유 | 검산(`D-3−3일소모+입고` vs `보고`) | 먼저 눈에 띄는 점 |
|---|---|---|---|---|---|---|
| 북곡 | AM-1 | 25.0 | 4.0 | 1.0 | 24 = 24 (일치) | 여유 1.0일 |
| 북곡 | DP-2 | 12.5 | 8.0 | 6.0 | **45 ≠ 40 (확인 후 사용)** | D일보유량_보고 불일치 |
| 북곡 | PG-3 | 8.3 | 12.0 | 11.0 | 120 = 120 (일치) | 여유 최댓값(11.0일) |
| 서림 | AM-1 | 16.7 | 6.0 | 3.0 | 18 = 18 (일치) | 담당자 B 소모 이견(비고) |
| 서림 | DP-2 | 33.3 | 3.0 | 1.0 | 12 = 12 (일치) | D일 소모 18시 집계 전 |
| 서림 | PG-3 | 50.0 | 2.0 | 1.0 | 30 = 30 (일치) | 구두 어림값('30 정도') |
| 해솔 | AM-1 | 50.0 | 2.0 | **-1.0** | 8 = 8 (일치) | 여유 최솟값(-1.0일, 고갈 후 도착) |
| 해솔 | DP-2 | 10.0 | 10.0 | 8.0 | 20 = 20 (일치) | 실사 확인 |
| 해솔 | PG-3 | 10.0 | 10.0 | 9.0 | 60 = 60 (일치) | 야간 소모 미반영 |

- **도전 과제 정답(`해솔 AM-1`, 일평균 6발·리드타임 3일)**: 보유 8발 → 소모율 `75.0%`, 보유일수 `1.3일`, 여유 `-1.7일` / 보유 4발 → 소모율 `150.0%`, 보유일수 `0.7일`, 여유 `-2.3일`.


In [ ]:
# [교수자용 검증 코드] 미션 1 — 대장 9행·검산 불일치(북곡 DP-2)·해솔 AM-1 음수 여유 확인
import csv
from pathlib import Path

with Path("02_데이터셋/거점_보유소모_대장.csv").open(encoding="utf-8") as f:
    next(f)
    rows = list(csv.DictReader(f))

assert len(rows) == 9
mismatches = []
for r in rows:
    expected = float(r["D-3일보유량"]) - (float(r["D-2일소모"]) + float(r["D-1일소모"]) + float(r["D일소모"])) + float(r["입고량_D-2~D"])
    if expected != float(r["D일보유량_보고"]):
        mismatches.append((r["거점"], r["품목"], expected, float(r["D일보유량_보고"])))

assert mismatches == [("북곡", "DP-2", 45.0, 40.0)]
haesol_am1 = [r for r in rows if r["거점"] == "해솔" and r["품목"] == "AM-1"][0]
assert round(float(haesol_am1["D일보유량_보고"]) / float(haesol_am1["일평균소모량_보고"]) - float(haesol_am1["리드타임_일"]), 1) == -1.0
print("[M1 검증 PASS] 9행 확인, 검산 불일치 1건(북곡 DP-2: 45 vs 40), 해솔 AM-1 여유 -1.0일 확인")
